# 05 · Köprü Testi, Duyarlılık Analizi ve Döngü 23 Senaryoları
**Tez:** Güneş Döngüleri 23–25 Boyunca Güneş Patlamaları, Güneş Lekeleri ve Jeomanyetik Etkiler
**Hazırlayan:** Tunahan İnci · **Defter sürümü:** 1.0 (05.11.2026)

**Durum:** 04 taraması, eski (GOES 8–12) ve yeni (GOES 13–19) uydular arasında tek ortak dönemin GOES-10 ile GOES-14'ün 2009 sonbaharındaki çakışması olduğunu doğruladı (GOES-14 2009-09-19'da başlıyor, GOES-10 2009-12-01'de bitiyor). Daha güçlü bir köprü yok.

**Literatür:** Hudson ve ark. (2023, *Solar Physics*), GOES-16 / GOES-15 oranını 243 patlamadan **1,10 ± 0,04** buluyor (bizim sonucumuz 1,08 — uyumlu). NOAA'ya göre GOES 8–15, 0.7 çarpanı kaldırıldıktan sonra doğru kalibre; aletler birbirleriyle **yaklaşık %20 içinde** karşılaştırılabilir.

**Bu defter:**
1. **Duyarlılık:** Kalibrasyonu yalnızca M sınıfı çiftlerle yeniden çözer, Döngü 25 / 24 oranının değişip değişmediğine bakar.
2. **1 dakikalık köprü:** GOES-10 ile GOES-14'ün çakışmasındaki **her dakikayı** karşılaştırır (11 patlama yerine on binlerce ölçüm).
3. **Döngü 23 senaryoları:** Eski grubun GOES-16 ölçeğine bağlanması için dört farklı varsayımla Döngü 25 / 23 oranını hesaplar. Sonuç tek sayı değil, savunulabilir bir aralık olarak raporlanacak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import re, json, hashlib, time, datetime as dt, requests
import xarray as xr
import matplotlib.pyplot as plt
from urllib.parse import urljoin
KOK = Path('/content/drive/MyDrive/gunes_tezi')
HAM, ISLENMIS, SEKIL = KOK/'data'/'raw', KOK/'data'/'processed', KOK/'figures'
(HAM/'ncei'/'avg1m').mkdir(parents=True, exist_ok=True)
print('Hazır')

In [ ]:
# Saf fonksiyonlar (test edilebilir)
import numpy as np, pandas as pd

def cift_olaylari(tum, esik=1e-6, aki='pik_aki'):
    # Aynı patlamayı (aynı grup) gören her uydu çifti için log akı oranı: ln(F_b / F_a), a < b
    t = tum[['grup', 'uydu', 'pik', aki]].rename(columns={aki: 'f'})
    m = t.merge(t, on='grup', suffixes=('_a', '_b'))
    m = m[(m.uydu_a < m.uydu_b) & (m.f_a >= esik) & (m.f_b >= esik)].copy()
    m['lr'] = np.log(m.f_b / m.f_a)
    m['pik'] = m.pik_a
    return m[['grup', 'pik', 'uydu_a', 'uydu_b', 'f_a', 'f_b', 'lr']].reset_index(drop=True)

def cift_ozeti(m):
    return (m.groupby(['uydu_a', 'uydu_b']).lr
              .agg(medyan='median', n='size',
                   iqr=lambda s: s.quantile(.75) - s.quantile(.25)).reset_index())

def kalibre_coz(ozet, referans=16, min_n=20):
    # Model: ln F_ölçülen(uydu) = ln F_gerçek + c_uydu.  Her çift için c_b - c_a = medyan(lr).
    # Ağırlıklı en küçük kareler, referans uydu c = 0. Referansa bağlanamayan bileşenler ayrıca raporlanır.
    s = ozet[ozet.n >= min_n]
    uydular = sorted(set(s.uydu_a) | set(s.uydu_b) | {referans})
    ebeveyn = {u: u for u in uydular}
    def bul(x):
        while ebeveyn[x] != x:
            ebeveyn[x] = ebeveyn[ebeveyn[x]]; x = ebeveyn[x]
        return x
    for a, b in zip(s.uydu_a, s.uydu_b):
        ebeveyn[bul(a)] = bul(b)
    bilesen = {}
    for u in uydular:
        bilesen.setdefault(bul(u), []).append(u)
    idx = {u: i for i, u in enumerate(uydular)}
    A, y, w = [], [], []
    for r in s.itertuples():
        row = np.zeros(len(uydular)); row[idx[r.uydu_b]] = 1; row[idx[r.uydu_a]] = -1
        A.append(row); y.append(r.medyan); w.append(np.sqrt(r.n))
    bagli = {}
    for kok, uyeler in bilesen.items():
        if referans in uyeler:
            capa = referans
        else:
            say = {u: s[(s.uydu_a == u) | (s.uydu_b == u)].n.sum() for u in uyeler}
            capa = max(say, key=say.get)
        row = np.zeros(len(uydular)); row[idx[capa]] = 1
        A.append(row); y.append(0.0); w.append(1e4)
        for u in uyeler:
            bagli[u] = referans in uyeler
    A, y, w = np.array(A), np.array(y), np.array(w)
    c = np.linalg.lstsq(A * w[:, None], y * w, rcond=None)[0]
    return pd.DataFrame({'uydu': uydular, 'c': c, 'faktor': np.exp(c),
                         'referansa_bagli': [bagli[u] for u in uydular]})

def bootstrap(m, referans=16, min_n=20, tekrar=200, tohum=42):
    rng = np.random.default_rng(tohum)
    gruplar = {k: g.lr.values for k, g in m.groupby(['uydu_a', 'uydu_b'])}
    sonuc = []
    for _ in range(tekrar):
        satir = [(a, b, np.median(rng.choice(v, len(v))), len(v), 0) for (a, b), v in gruplar.items()]
        oz = pd.DataFrame(satir, columns=['uydu_a', 'uydu_b', 'medyan', 'n', 'iqr'])
        sonuc.append(kalibre_coz(oz, referans, min_n).set_index('uydu').faktor)
    b = pd.concat(sonuc, axis=1)
    return pd.DataFrame({'faktor_q025': b.quantile(.025, axis=1), 'faktor_q975': b.quantile(.975, axis=1)})

def kalibrasyonu_uygula(ncei, katsayi):
    f = ncei.uydu.map(katsayi.set_index('uydu').faktor).fillna(1.0)
    out = ncei.copy()
    out['kal_faktor'] = f
    out['pik_aki_kal'] = out.pik_aki / f
    return out

In [ ]:
# Ortak veriler ve döngü sayım fonksiyonu
tum = pd.read_parquet(ISLENMIS/'flare_ncei_tum_uydular.parquet')
ncei = pd.read_parquet(ISLENMIS/'flare_ncei.parquet')
ana = pd.read_csv(ISLENMIS/'03_kalibrasyon_katsayilari.csv')
sn = pd.read_parquet(ISLENMIS/'sn_aylik_1749.parquet').set_index('tarih').sn
DONGU = {'23': ('2000-01-01', '2008-11-30'), '24': ('2008-12-01', '2019-11-30'), '25': ('2019-12-01', '2025-12-31')}
ESKI = [8, 9, 10, 11, 12]

def dongu_orani(faktor_dict):
    f = ncei.uydu.map(faktor_dict).fillna(1.0)
    aki = ncei.pik_aki / f
    sonuc = {}
    for d, (b, s) in DONGU.items():
        secim = (ncei.pik >= b) & (ncei.pik <= s) & (aki >= 1e-5)
        sonuc[d] = secim.sum() / sn[b:s].sum() * 1000
    return sonuc

taban = dongu_orani(dict(zip(ana.uydu, ana.faktor)))
print('Ana çözüm, 1000 SN başına M+X:', {k: round(v, 1) for k, v in taban.items()})

## 1 · Duyarlılık: yalnızca M sınıfı çiftlerle kalibrasyon
Döngü sayımları M1 eşiğine dayandığı için, kalibrasyonun tam bu akı düzeyinde doğru olması gerekir. C sınıfı ağırlıklı ana çözüm ile M sınıfı çözüm karşılaştırılır.

In [ ]:
m_all = cift_olaylari(tum, esik=1e-6)
m_M = m_all[np.sqrt(m_all.f_a * m_all.f_b) >= 1e-5]
oz_M = cift_ozeti(m_M)
print('M sınıfı çiftler:'); print(oz_M.assign(oran=np.exp(oz_M.medyan)).round(3).to_string(index=False))
cozM = kalibre_coz(oz_M, referans=16, min_n=10)
karsilastir = ana[['uydu', 'faktor']].merge(cozM[['uydu', 'faktor', 'referansa_bagli']], on='uydu', how='outer', suffixes=('_ana', '_M'))
print('\nKatsayılar, ana (≥C1) ve yalnızca M:'); print(karsilastir.round(3).to_string(index=False))

fM = dict(zip(ana.uydu, ana.faktor)); fM.update({u: f for u, f, b in zip(cozM.uydu, cozM.faktor, cozM.referansa_bagli) if b and u not in ESKI})
dM = dongu_orani(fM)
print(f"\nDöngü 25 / 24 — ana: {taban['25']/taban['24']:.3f} | yalnızca M kalibrasyonu: {dM['25']/dM['24']:.3f}")

## 2 · 1 dakikalık köprü: GOES-10 ve GOES-14
2009-09-19 ile 2009-12-01 arasındaki günlük 1 dakikalık dosyalar indirilir (her uydu için ~75 dosya, küçük boyutlu). İki uydunun aynı dakikadaki XRS-B akısı karşılaştırılır.

Dikkat: Bu dönem güneş minimumu. Akılar çoğunlukla A ve B düzeyinde, yani patlama sayımında kullandığımız M düzeyinin 100–1000 kat altında. Sonuç bu yüzden **zayıf ama bağımsız** bir köprü tahmini.

In [ ]:
TEMEL = 'https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/xrs/'
BAS_K, SON_K = pd.Timestamp('2009-09-19'), pd.Timestamp('2009-12-01')

def linkler(url):
    try:
        r = requests.get(url, timeout=60); r.raise_for_status()
        return [urljoin(url, h) for h in re.findall(r'href="([^"?#]+)"', r.text) if not h.startswith(('/', 'http', '..'))]
    except Exception as e:
        print('  erişilemedi:', url, e); return []

def gunluk_dosyalar(uydu):
    kok = f'{TEMEL}goes{uydu:02d}/'
    kuyruk = [l for l in linkler(kok) if 'avg1m' in l.lower() and l.endswith('/')]
    bulunan = []
    while kuyruk:
        u = kuyruk.pop()
        for l in linkler(u):
            if l.endswith('/') and l.startswith(u) and l != u and ('2009' in l[len(u):] or not re.search(r'\d{4}', l[len(u):])):
                kuyruk.append(l)
            m = re.search(r'_d(\d{8})_', l)
            if m and l.endswith('.nc') and BAS_K <= pd.Timestamp(m.group(1)) <= SON_K:
                bulunan.append(l)
    return sorted(bulunan)

indirilen = {}
for uydu in (10, 14):
    urls = gunluk_dosyalar(uydu)
    print(f'GOES-{uydu}: {len(urls)} günlük dosya bulundu')
    yollar = []
    for u in urls:
        p = HAM/'ncei'/'avg1m'/u.rsplit('/', 1)[1]
        if not p.exists():
            for i in range(3):
                try:
                    r = requests.get(u, timeout=120); r.raise_for_status(); p.write_bytes(r.content); break
                except Exception as e:
                    print('  !', e); time.sleep(5)
        yollar.append(p)
    indirilen[uydu] = yollar

ornek = xr.open_dataset(indirilen[10][0])
print('\nGOES-10 dosya değişkenleri:'); print(ornek)
ornek.close()

In [ ]:
def xrsb_oku(yollar):
    parca = []
    for p in yollar:
        with xr.open_dataset(p) as ds:
            akı = [v for v in ds.data_vars if 'xrsb' in v.lower() and 'flux' in v.lower() and 'flag' not in v.lower()]
            bayrak = [v for v in ds.data_vars if 'xrsb' in v.lower() and ('flag' in v.lower() or 'quality' in v.lower())]
            secim = akı[:1] + bayrak[:1]
            d = ds[secim].to_dataframe().reset_index()
            d = d.rename(columns={akı[0]: 'f'} | ({bayrak[0]: 'bayrak'} if bayrak else {}))
        parca.append(d)
    d = pd.concat(parca, ignore_index=True)
    print('  kullanılan değişkenler:', secim)
    return d

g10 = xrsb_oku(indirilen[10]); g14 = xrsb_oku(indirilen[14])
for ad, g in (('GOES-10', g10), ('GOES-14', g14)):
    if 'bayrak' in g: print(ad, 'bayrak değerleri:', g.bayrak.value_counts().head(6).to_dict())

k = g10.merge(g14, on='time', suffixes=('_10', '_14'))
if 'bayrak_10' in k: k = k[k.bayrak_10 == 0]
if 'bayrak_14' in k: k = k[k.bayrak_14 == 0]
k = k[(k.f_10 > 0) & (k.f_14 > 0)].copy()
k['oran'] = k.f_14 / k.f_10
k['duzey'] = pd.cut(np.sqrt(k.f_10 * k.f_14), [1e-9, 1e-8, 1e-7, 1e-6, 1], labels=['<A1', 'A', 'B', '≥C'], right=False)
print(f'\nOrtak geçerli dakika: {len(k):,}')
print('Akı düzeyine göre GOES-14 / GOES-10 medyan oran:')
print(k.groupby('duzey', observed=True).oran.agg(['median', 'size']).round(3).to_string())
print('\nHaftalık medyan oran (B ve üstü):')
print(k[k.f_10 >= 1e-7].set_index('time').oran.resample('W').agg(['median', 'size']).round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
ax[0].loglog(k.f_10, k.f_14, ',', alpha=.3); lim = [k[['f_10', 'f_14']].min().min(), k[['f_10', 'f_14']].max().max()]
ax[0].plot(lim, lim, 'k:', lw=1); ax[0].set_xlabel('GOES-10 XRS-B (W/m²)'); ax[0].set_ylabel('GOES-14 XRS-B (W/m²)')
ax[1].plot(k.time, k.oran, ',', alpha=.3); ax[1].set_ylim(0.5, 2); ax[1].axhline(1, color='k', ls=':')
ax[1].set_ylabel('GOES-14 / GOES-10'); fig.suptitle('1 dakikalık köprü, 2009 sonbaharı')
fig.savefig(SEKIL/'05_kopru_1dk.png', dpi=200); plt.show()

kopru_1dk_B = k[k.f_10 >= 1e-7].oran.median()
kopru_1dk_A = k[(k.f_10 >= 1e-8) & (k.f_10 < 1e-7)].oran.median()
print(f'\nKöprü oranı GOES-14/GOES-10 — B ve üstü: {kopru_1dk_B:.3f} | A düzeyi: {kopru_1dk_A:.3f}')

## 3 · Döngü 23 senaryoları
Eski grubun kendi içindeki oranları (03 ana çözümü) sabit tutulur. Değişen tek şey **GOES-10'un GOES-16 ölçeğindeki katsayısı**:

| Senaryo | GOES-10 katsayısı | Dayanak |
|---|---|---|
| S0 · Bağlantısız | 1,000 | Eski grup GOES-16 ile aynı ölçekte varsayılır |
| S1 · Literatür | GOES-15 ile aynı (0,936) | NOAA: GOES 8–15 tutarlı kalibre (Hudson ve ark. 2023) |
| S2 · Patlama köprüsü | 0,776 | 03 defteri, 11 B sınıfı patlama |
| S3 · Dakika köprüsü | GOES-14 / oran | Bu defter, Bölüm 2 (B ve üstü) |

In [ ]:
f14 = float(ana.set_index('uydu').faktor[14]); f15 = float(ana.set_index('uydu').faktor[15])
ic = ana.set_index('uydu').faktor.reindex(ESKI).dropna()   # GOES-10'a göre (ana çözümde GOES-10 = 1)
senaryo = {'S0 Bağlantısız': 1.0, 'S1 Literatür (=GOES-15)': f15,
           'S2 Patlama köprüsü (2009, 11 olay)': f14 / 1.107, 'S3 Dakika köprüsü (2009, B+)': f14 / kopru_1dk_B}
satir = []
for ad, a10 in senaryo.items():
    fd = dict(zip(ana.uydu, ana.faktor)); fd.update({u: a10 * v for u, v in ic.items()})
    d = dongu_orani(fd)
    satir.append([ad, round(a10, 3), round(d['23'], 1), round(d['24'], 1), round(d['25'], 1),
                  round(d['25'] / d['23'], 3), round(d['25'] / d['24'], 3)])
sen = pd.DataFrame(satir, columns=['Senaryo', 'GOES-10 katsayısı', 'D23 /1000SN', 'D24 /1000SN', 'D25 /1000SN', 'D25/D23', 'D25/D24'])
sen.to_csv(ISLENMIS/'05_dongu23_senaryolar.csv', index=False)
sen